# Drug approval rates & indication frequencies

Exploration of the **`ours_di`** drug-indication dataset and the saved model predictions.
Each row is one *(drug, indication)* pair; `label = 1` means that pair eventually reached approval.

Three questions:
1. **How many drugs were tested against multiple indications?**
2. **What is the distribution of approvals across indications, for each drug?**
3. **How well does prediction performance discriminate between different indications?**

Run top-to-bottom. Uses the repo's `.venv` (pandas / numpy / matplotlib / scikit-learn).

In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

# repo root = parent of this notebook's dir (dsm/)
ROOT = Path.cwd()
if ROOT.name == "dsm":
    ROOT = ROOT.parent
DATA = ROOT / "data" / "datasets" / "ours_di.parquet"
RUNS = ROOT / "runs"

pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (8, 4.5)
print("data:", DATA)
assert DATA.exists(), f"missing {DATA} — run `python -m dsm materialize ours_di`"

In [ ]:
# --- load the dataset, keep only the columns this analysis needs ---
raw = pd.read_parquet(DATA)

df = raw[["example_id", "label", "phase", "split", "drug_name", "indication",
         "mesh_indication", "drugbank_id", "icd_codes", "disease_area"]].copy()

# Drug identity = the part of example_id before '__' (matches the pipeline's seen/unseen
# stratification). indication = the part after. Fall back to the explicit columns where useful.
df["drug_key"] = df["example_id"].str.split("__").str[0]
df["indication"] = df["indication"].fillna(df["example_id"].str.split("__").str[1])
df["label"] = df["label"].astype(int)

print(f"{len(df):,} (drug, indication) rows")
print(f"{df['drug_key'].nunique():,} distinct drugs")
print(f"{df['indication'].nunique():,} distinct raw indications")
print(f"{df['mesh_indication'].nunique():,} distinct MeSH-grouped indications")
print(f"overall approval rate: {df['label'].mean():.3f}")
df.head()

### A robust disease area: ICD-10 chapters

The dataset's `disease_area` column is LLM-derived (see `disease_confidence` / `disease_reasoning`)
and is `unknown`/`other` for ~41% of rows — not a dependable grouping. Instead we derive the disease
area directly from each row's **ICD-10 codes**, mapped to the fixed 22-chapter ICD-10 taxonomy. The
chapter is fully determined by the leading letter (+ the 2-digit number for the C/D and H splits), so
it's deterministic and covers every row. Each row gets the *modal* chapter across its codes.

In [ ]:
# ICD-10 chapter boundaries. Most chapters = the leading letter; C/D and H split on the 2-digit
# category number (C00-C97 + D00-D49 = Neoplasms, D50-D89 = Blood; H00-H59 = Eye, H60-H95 = Ear).
def icd_chapter(code):
    code = str(code).strip().upper()
    if not code or not code[0].isalpha():
        return None
    L = code[0]
    try:
        num = int(code[1:3])
    except ValueError:
        num = 0
    if L in "AB":   return "Infectious & parasitic"
    if L == "C" or (L == "D" and num <= 49): return "Neoplasms"
    if L == "D":    return "Blood & immune"
    if L == "E":    return "Endocrine & metabolic"
    if L == "F":    return "Mental & behavioral"
    if L == "G":    return "Nervous system"
    if L == "H":    return "Eye & adnexa" if num <= 59 else "Ear & mastoid"
    if L == "I":    return "Circulatory"
    if L == "J":    return "Respiratory"
    if L == "K":    return "Digestive"
    if L == "L":    return "Skin & subcutaneous"
    if L == "M":    return "Musculoskeletal"
    if L == "N":    return "Genitourinary"
    if L == "O":    return "Pregnancy & childbirth"
    if L == "P":    return "Perinatal"
    if L == "Q":    return "Congenital"
    if L == "R":    return "Symptoms & abnormal findings"
    if L in "ST":   return "Injury & poisoning"
    if L in "VWXY": return "External causes"
    if L == "Z":    return "Health-status factors"
    if L == "U":    return "Special purposes"
    return None

def modal_chapter(codes):
    chaps = [c for c in (icd_chapter(x) for x in codes) if c]
    return Counter(chaps).most_common(1)[0][0] if chaps else None

df["icd_chapter"] = df["icd_codes"].map(modal_chapter)
n_null = int(df["icd_chapter"].isna().sum())
print(f"ICD-10 chapter assigned to {len(df) - n_null:,}/{len(df):,} rows "
      f"({n_null} unmapped) across {df['icd_chapter'].nunique()} chapters")
print(f"(compare: disease_area is unknown/other for "
      f"{df['disease_area'].isin(['unknown', 'other']).mean():.0%} of rows)\n")

# Approval rate by chapter — a robust, deterministic disease-area frequency/outcome view.
by_chap = (df.groupby("icd_chapter")
             .agg(n=("example_id", "size"), n_approved=("label", "sum"))
             .assign(approval_rate=lambda d: d.n_approved / d.n)
             .sort_values("n", ascending=False))

ch = by_chap.iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
axes[0].barh(ch.index, ch["n"], color="#4c72b0")
axes[0].set_title("Rows per ICD-10 chapter"); axes[0].set_xlabel("# (drug,indication) rows")
axes[1].barh(ch.index, ch["approval_rate"], color="#55a868")
axes[1].axvline(df["label"].mean(), ls="--", c="k", lw=1, label="overall rate")
axes[1].set_title("approval rate by chapter"); axes[1].set_xlabel("approval rate"); axes[1].legend()
axes[1].set_yticklabels([])
plt.tight_layout(); plt.show()
by_chap

## 1. How many drugs were tested against multiple indications?

Group by `drug_key` and count distinct indications per drug.

In [ ]:
per_drug = (df.groupby("drug_key")
              .agg(n_indications=("indication", "nunique"),
                   n_rows=("example_id", "size"),
                   n_approved=("label", "sum"))
              .reset_index())
per_drug["approval_rate"] = per_drug["n_approved"] / per_drug["n_rows"]

n_drugs = len(per_drug)
n_multi = int((per_drug["n_indications"] > 1).sum())
print(f"drugs total:                 {n_drugs:,}")
print(f"drugs with >1 indication:    {n_multi:,}  ({n_multi / n_drugs:.1%})")
print(f"drugs with exactly 1:        {n_drugs - n_multi:,}  ({(n_drugs - n_multi) / n_drugs:.1%})")
print(f"max indications for one drug: {per_drug['n_indications'].max()}")
print("\nindications-per-drug distribution:")
print(per_drug["n_indications"].describe())

In [ ]:
# Histogram of indications-per-drug (clipped at 10+ for readability)
clip = per_drug["n_indications"].clip(upper=10)
counts = clip.value_counts().sort_index()
labels = [str(int(i)) for i in counts.index]
if counts.index.max() == 10:
    labels[-1] = "10+"

fig, ax = plt.subplots()
ax.bar(labels, counts.values, color="#4c72b0")
ax.set_xlabel("# distinct indications a drug was tested against")
ax.set_ylabel("# drugs")
ax.set_title("Indications per drug")
for x, v in zip(labels, counts.values):
    ax.text(x, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()

print("Top 10 most-tested drugs:")
top = per_drug.sort_values("n_indications", ascending=False).head(10).merge(
    df[["drug_key", "drug_name"]].drop_duplicates("drug_key"), on="drug_key", how="left")
top[["drug_name", "drug_key", "n_indications", "n_approved", "approval_rate"]]

## Indication frequency

Which indications show up most often, and how does their base approval rate vary? (For a robust,
deterministic disease-area view, see the ICD-10-chapter chart above; this slices on the more granular
MeSH-grouped indication string.)

In [ ]:
GROUP = "mesh_indication"   # switch to "indication" for raw, ungrouped indication strings

per_ind = (df.dropna(subset=[GROUP])
             .groupby(GROUP)
             .agg(n=("example_id", "size"), n_approved=("label", "sum"))
             .reset_index())
per_ind["approval_rate"] = per_ind["n_approved"] / per_ind["n"]
per_ind = per_ind.sort_values("n", ascending=False)

topN = per_ind.head(20).iloc[::-1]   # most frequent at top of barh
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].barh(topN[GROUP], topN["n"], color="#4c72b0")
axes[0].set_title(f"20 most frequent indications ({GROUP})"); axes[0].set_xlabel("# (drug,indication) rows")
axes[1].barh(topN[GROUP], topN["approval_rate"], color="#55a868")
axes[1].axvline(df["label"].mean(), ls="--", c="k", lw=1, label="overall rate")
axes[1].set_title("approval rate of those indications"); axes[1].set_xlabel("approval rate"); axes[1].legend()
axes[1].set_yticklabels([])
plt.tight_layout(); plt.show()

print("Most frequent indications:")
per_ind.head(15)

## 2. Distribution of approvals across indications, per drug

For each drug, the approval rate across the indications it was tested against. A drug at 0.0 failed
every indication; at 1.0 it succeeded in all; in between, it's *indication-specific* — the
interesting case where which indication matters.

In [ ]:
multi = per_drug[per_drug["n_indications"] > 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
# (a) per-drug approval rate across all drugs
axes[0].hist(per_drug["approval_rate"], bins=20, color="#4c72b0", edgecolor="white")
axes[0].set_title("Per-drug approval rate (all drugs)")
axes[0].set_xlabel("fraction of a drug's indications approved"); axes[0].set_ylabel("# drugs")
# (b) restricted to multi-indication drugs — shows how many are 'mixed' (0<rate<1)
axes[1].hist(multi["approval_rate"], bins=20, color="#c44e52", edgecolor="white")
axes[1].set_title(f"Per-drug approval rate (>1 indication, n={len(multi):,})")
axes[1].set_xlabel("fraction of a drug's indications approved"); axes[1].set_ylabel("# drugs")
plt.tight_layout(); plt.show()

n_all_fail = int((multi["approval_rate"] == 0).sum())
n_all_ok = int((multi["approval_rate"] == 1).sum())
n_mixed = len(multi) - n_all_fail - n_all_ok
print(f"Among {len(multi):,} multi-indication drugs:")
print(f"  failed ALL indications:    {n_all_fail:,}  ({n_all_fail/len(multi):.1%})")
print(f"  approved in ALL:           {n_all_ok:,}  ({n_all_ok/len(multi):.1%})")
print(f"  MIXED (some yes, some no): {n_mixed:,}  ({n_mixed/len(multi):.1%})  <- indication-specific outcomes")

In [ ]:
# A concrete look at a few 'mixed' multi-indication drugs: same molecule, different fates.
mixed_keys = multi[(multi["approval_rate"] > 0) & (multi["approval_rate"] < 1)] \
                 .sort_values("n_indications", ascending=False)["drug_key"].head(3)
for k in mixed_keys:
    sub = df[df["drug_key"] == k]
    name = sub["drug_name"].iloc[0]
    print(f"\n=== {name} ({k}) — {len(sub)} indications, {sub['label'].sum()} approved ===")
    print(sub[["indication", "phase", "label"]].sort_values("label", ascending=False).to_string(index=False))

## 3. How well does prediction performance discriminate between indications?

Two complementary views, using the saved test-set predictions:

- **(A) Across disease areas** — does the model score *better* for some areas than others?
  (Per-ICD-10-chapter ROC-AUC.)
- **(B) Within a drug** — for a drug with mixed outcomes across indications, can the model rank its
  *approved* indication above its *failed* one? This is the discrimination ChemAP (SMILES-only)
  structurally cannot do — it assigns one probability per molecule — but a disease-aware model can.

In [ ]:
# Load whatever prediction runs exist, joined to indication metadata (test rows only).
CANDIDATES = ["xgb_di_2019", "xgb_di_md", "hint_di_2019", "chemap_di_2019"]
meta = df[["example_id", "drug_key", "drug_name", "indication", "icd_chapter", GROUP]]

preds = {}
for name in CANDIDATES:
    p = RUNS / name / "predictions.parquet"
    if not p.exists():
        print(f"(skip {name} — no predictions)"); continue
    pr = pd.read_parquet(p).merge(meta, on="example_id", how="left")
    preds[name] = pr
    print(f"{name:16s} n={len(pr):5d}  ROC-AUC={roc_auc_score(pr.label, pr.y_proba):.4f}  "
          f"PR-AUC={average_precision_score(pr.label, pr.y_proba):.4f}")
assert preds, "no prediction runs found — run e.g. `python -m dsm run xgb_di_2019`"

In [ ]:
# (A) Per-group ROC-AUC over the robust ICD-10-chapter disease area. (mesh_indication is too
# granular for the ~2.6k-row test set — most groups would be too small; bump GROUP_BY to a finer
# column and lower MIN_N only if you accept noisier per-group estimates.)
GROUP_BY, MIN_N, MIN_POS = "icd_chapter", 25, 3

def per_group_auc(pr, col):
    out = []
    for grp, g in pr.dropna(subset=[col]).groupby(col):
        npos = int(g.label.sum())
        if len(g) >= MIN_N and MIN_POS <= npos <= len(g) - MIN_POS:
            out.append({col: grp, "n": len(g), "n_pos": npos,
                        "base_rate": g.label.mean(), "roc_auc": roc_auc_score(g.label, g.y_proba)})
    return pd.DataFrame(out).sort_values("roc_auc", ascending=False)

MODEL = "xgb_di_2019" if "xgb_di_2019" in preds else next(iter(preds))
grp_auc = per_group_auc(preds[MODEL], GROUP_BY)
print(f"model={MODEL}: {len(grp_auc)} {GROUP_BY} groups with >={MIN_N} test rows & both classes")
print(f"per-group ROC-AUC: mean={grp_auc.roc_auc.mean():.3f}  "
      f"min={grp_auc.roc_auc.min():.3f}  max={grp_auc.roc_auc.max():.3f}")

fig, ax = plt.subplots(figsize=(9, 6))
gp = grp_auc.sort_values("roc_auc")
colors = ["#c44e52" if v < 0.5 else "#4c72b0" for v in gp["roc_auc"]]
ax.barh(gp[GROUP_BY], gp["roc_auc"], color=colors)
ax.axvline(0.5, ls="--", c="k", lw=1, label="chance")
ax.set_xlabel("ROC-AUC"); ax.set_title(f"Discrimination by {GROUP_BY} ({MODEL})"); ax.legend()
for y, (v, n) in enumerate(zip(gp["roc_auc"], gp["n"])):
    ax.text(v, y, f" {v:.2f} (n={n})", va="center", fontsize=8)
plt.tight_layout(); plt.show()

grp_auc

In [ ]:
# (B) Within-drug discrimination. For drugs that appear >1x in the TEST set with a mix of
# approved/failed indications, can the model rank the approved indication above the failed ones?
# Metric: per-drug within-pair AUC (prob of scoring a random approved indication above a random
# failed one, for that same drug). 0.5 = no within-drug signal; constant-per-drug models = 0.5.
def within_drug_auc(pr):
    rows = []
    for key, g in pr.groupby("drug_key"):
        pos, neg = g[g.label == 1], g[g.label == 0]
        if len(pos) and len(neg):
            # fraction of (approved, failed) pairs the model orders correctly (ties = 0.5)
            diff = pos.y_proba.values[:, None] - neg.y_proba.values[None, :]
            acc = (diff > 0).mean() + 0.5 * (diff == 0).mean()
            rows.append({"drug_key": key, "n_ind": len(g), "n_pos": len(pos),
                         "within_auc": acc, "prob_spread": g.y_proba.max() - g.y_proba.min()})
    return pd.DataFrame(rows)

summary = []
for name, pr in preds.items():
    w = within_drug_auc(pr)
    summary.append({"model": name, "n_mixed_drugs": len(w),
                    "mean_within_drug_auc": w["within_auc"].mean(),
                    "mean_prob_spread": w["prob_spread"].mean()})
summary = pd.DataFrame(summary)
print("Within-drug discrimination (drugs with mixed outcomes in the test set):")
print("  mean_within_drug_auc 0.5 = cannot tell a drug's indications apart; >0.5 = ranks the")
print("  approved indication higher. mean_prob_spread = how much a model's probs vary across")
print("  one drug's indications (0 => constant per molecule, e.g. SMILES-only models).")
summary

In [ ]:
# Visualize within-drug probability spread per model: a model that ignores indication has a
# spike at spread=0; a disease-aware model spreads its probabilities across a drug's indications.
models = list(preds)
fig, axes = plt.subplots(1, len(models), figsize=(4.2 * len(models), 4), sharey=True)
if len(models) == 1:
    axes = [axes]
for ax, name in zip(axes, models):
    w = within_drug_auc(preds[name])
    ax.hist(w["prob_spread"], bins=20, color="#8172b3", edgecolor="white")
    ax.set_title(f"{name}\nmean within-drug AUC={w['within_auc'].mean():.3f}")
    ax.set_xlabel("prob spread across a drug's indications")
axes[0].set_ylabel("# multi-indication drugs (test)")
plt.tight_layout(); plt.show()

### Takeaways

- **(1)** the indications-per-drug bar chart + the `>1 indication` percentage answer how many drugs
  are multi-indication.
- **(2)** the per-drug approval-rate histograms show how often a drug's fate is *indication-specific*
  (the `MIXED` bucket) versus all-or-nothing.
- **(3A)** ROC-AUC by **ICD-10 chapter** (a deterministic disease area derived from each row's codes,
  not the LLM `disease_area`) shows the model is much stronger in some therapeutic areas than others.
- **(3B)** within-drug AUC + probability spread show whether a model can separate a single drug's
  good vs bad indications. A SMILES-only model (`chemap`) gives one probability per molecule →
  spread ≈ 0 and within-drug AUC ≈ 0.5; disease-aware models (`xgb`, `hint`) should do better, which
  is exactly the per-indication signal those models add over structure alone.